<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Estimación de la calidad relativa de los productos

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.
- `funciones.estimate_quality`: estimación de la calidad por valores unitarios ajustados.

In [1]:
import pandas as pd
import numpy as np
from funciones.estimate_quality import estimate_quality as quality

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, temp

### Lectura de datos

Carga la base de comercio bilateral y la información complementaria de distancia entre
capitales (CEPII) y producto interno bruto per cápita de los países exportadores (FMI).

In [3]:
df_comercio = pd.read_parquet(temp / 'df_comercio.parquet')
df_info_calidad = pd.read_parquet(raw / 'comercio' / 'df_info_calidad.parquet').drop(columns=['distw'])

### Estandarización de los códigos ISO

Corrige las inconsistencias de codificación frente a la base de comercio: `ROU` pasa a `ROM`
y `SRB` pasa a `SER`. Renombra las columnas de país para que coincidan con las del comercio.

In [4]:
df_info_calidad['OrigenISO'] = np.where(df_info_calidad['OrigenISO'] == 'ROU', 'ROM', df_info_calidad['OrigenISO'])
df_info_calidad['DestinoISO'] = np.where(df_info_calidad['DestinoISO'] == 'ROU', 'ROM', df_info_calidad['DestinoISO'])

df_info_calidad['OrigenISO'] = np.where(df_info_calidad['OrigenISO'] == 'SRB', 'SER', df_info_calidad['OrigenISO'])
df_info_calidad['DestinoISO'] = np.where(df_info_calidad['DestinoISO'] == 'SRB', 'SER', df_info_calidad['DestinoISO'])

df_info_calidad = df_info_calidad.rename(columns={'OrigenISO':'ISO Exportador','DestinoISO':'ISO Importador'})

temp.mkdir(parents=True, exist_ok=True)
df_info_calidad.to_parquet(temp / 'df_info_calidad.parquet', index=False)

### Calidad estimada por producto

Estima la calidad relativa mediante valores unitarios ajustados por distancia e ingreso.

El ejercicio se realiza dos veces sobre particiones complementarias de la base: el año 2013 por
separado y el resto de los años en conjunto. Ambas corridas usan el cuantil de frontera 0,9, por
lo que se define una tabla de parámetros con el filtro y el archivo de salida de cada una.

In [5]:
# tabla de parámetros: filtro sobre el año y archivo de salida.
# el cuantil de frontera es 0.9 en ambos casos, que es el valor por defecto de la función
parametros_calidad = [
    ("Año != 2013", 'df_calidad.parquet'),
    ("Año == 2013", 'df_calidad_13.parquet'),
]

temp.mkdir(parents=True, exist_ok=True)
for filtro, salida in parametros_calidad:
    df_resultado = quality(
        df_comercio
        .query(filtro)
        .merge(df_info_calidad, how='left', on=['Año','ISO Importador','ISO Exportador'])
        .dropna(),
        frontier_quantile=0.9)
    df_resultado.to_parquet(temp / salida, index=False)
    print(f"{salida}: {df_resultado.shape[0]:,} filas")

df_calidad.parquet: 5,139,564 filas


df_calidad_13.parquet: 1,530,831 filas
